# Demographics table

**Purpose:** Generate participant summary statistics stratified by cognitive status.

**Expected inputs**
- `../data/plasma_metadata_matched_all_outcomes.tsv`

**Main outputs**
- `Demographics summary table for manuscript Table 1`

> Notes for reuse: data files are not included in this repository. Update paths in the cells below to match the local location of the approved, de-identified data release. Notebook outputs have been cleared for public sharing.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
DATA_DIR = PROJECT_ROOT / 'data'
OUTPUT_DIR = PROJECT_ROOT / 'output_files'
FIGURE_DIR = PROJECT_ROOT / 'figures'

for directory in [DATA_DIR, OUTPUT_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests


In [ ]:
# And metadata
md = pd.read_csv('../data/plasma_metadata_matched_all_outcomes.tsv', sep = '\t', dtype={'UCSD': str, 'Unnamed: 0': str, 'sample_name': str}).set_index('sample_name')


In [ ]:
md['site'].isna().sum()


In [ ]:
md['Race: White'] = md['RACE'].replace({2: 0, 3: 0, 5: 0, 50:0})


In [ ]:
md['Race: Black or African American'] = md['RACE'].replace({2:1, 1: 0, 3: 0, 5: 0, 50:0})


In [ ]:
md['Race: Black or African American'].value_counts()


In [ ]:
md['Race: Other'] = md['RACE'].replace({1:0, 2:0, 3: 1, 5: 1, 50:1})


In [ ]:
md['Ethnicity: Hispanic'] = md['HISPANIC'].replace({9:0})


In [ ]:
md['Ethnicity: Hispanic'].value_counts()


In [ ]:
md['Sex: Female'] = md['SEX'].replace({1: 0, 2: 1})


In [ ]:
md['Sex: Female'].value_counts()


In [ ]:
md['Sex: Female'] = md['SEX'].replace({1: 0, 2: 1})


In [ ]:
md['Age'] = md['NACCAGE']
md['BMI'] = md['NACCBMI']


In [ ]:
md['Age'].min()


In [ ]:
md['Age'].std()


In [ ]:
md['BMI'] = md['BMI'].replace(np.inf, np.nan)


In [ ]:
def make_group_summary_table(
    df: pd.DataFrame,
    group_col: str,
    variables: list,
    var_types: dict,
    group_order: list = None,
    continuous_test: str = "anova",  # "anova" or "kruskal"
    digits_mean: int = 3,
    digits_sd: int = 3,
    digits_pct: int = 1,
    digits_stat: int = 3,
    digits_p: int = 3,
):
    """
    Build a Table 1 by group with stats and p-values.

    Parameters
    ----------
    df : DataFrame
        Input data (one row per participant).
    group_col : str
        Column giving group (group).
    variables : list[str]
        Variables to summarize, in the order you want them displayed.
        Include a special string "#N" to insert the "No. of participants" row.
    var_types : dict[str, str]
        Map of variable -> "continuous" or "categorical".
        (For binary categorical, just ensure the column is categorical/boolean or has up to 2 levels.)
    group_order : list[str], optional
        Order of countries (columns). If None, inferred from the data.
    continuous_test : {"anova","kruskal"}
        Which test for continuous variables.
    digits_* : int
        Formatting precision.

    Returns
    -------
    pd.DataFrame
        Wide table with columns: Parameter, <group1>, <group2>, ..., Statistic, P-value
    """
    d = df.copy()
    # ensure group is categorical with desired order
    if group_order is None:
        group_order = list(pd.Series(d[group_col]).dropna().unique())
    d[group_col] = pd.Categorical(d[group_col], categories=group_order, ordered=True)

    # helper formatters
    def fmt_mean_sd(x):
        return f"{np.nanmean(x):.{digits_mean}f} ± {np.nanstd(x, ddof=1):.{digits_sd}f}"

    def fmt_n_pct(n, denom):
        pct = 100 * (n / denom) if denom > 0 else np.nan
        return f"{int(n)} ({pct:.{digits_pct}f}%)"

    def fmt_stat_p(stat, p):
        stat_s = f"{stat:.{digits_stat}f}" if pd.notnull(stat) else ""
        if p is None or np.isnan(p):
            p_s = ""
        elif p < 10**(-digits_p):
            p_s = f"<1e-{digits_p}"
        else:
            p_s = f"{p:.{digits_p}g}"
        return stat_s, p_s

    rows = []
    raw_pvals = []        # <--- COLLECT RAW P-VALUES HERE
    row_indices = []      # <--- KEEP TRACK OF WHICH ROW THEY BELONG TO
    
    # Top row: number of participants per group
    if "#N" in variables:
        total_n = len(d)
        row = {"Parameter": "No. of participants"}
        for c in group_order:
            n_c = (d[group_col] == c).sum()
            row[c] = fmt_n_pct(n_c, total_n)
        row["Statistic"], row["P-value"] = "", ""
        rows.append(row)

    for var in variables:
        if var == "#N":
            continue

        vtype = var_types.get(var, "categorical")  # default to categorical
        row = {"Parameter": var}

        # group-wise subsets
        groups = [d.loc[d[group_col] == c, var] for c in group_order]

        if vtype == "continuous":
            # per-group mean ± SD using non-missing n within each group
            for c, g in zip(group_order, groups):
                row[c] = fmt_mean_sd(g.astype(float))

            # test across countries
            if continuous_test == "anova":
                # f_oneway requires at least 2 non-empty groups
                non_empty = [g.dropna().astype(float) for g in groups if g.dropna().shape[0] > 1]
                if len(non_empty) >= 2:
                    stat, p = stats.f_oneway(*non_empty)
                else:
                    stat, p = np.nan, np.nan
                stat_s, p_s = fmt_stat_p(stat, p)
                row["Statistic"] = stat_s
                row["P-value"] = p_s
                
                # store raw p for correction
                raw_pvals.append(p)
                row_indices.append(len(rows))   # store row index
                rows.append(row)
                
            else:
                # Kruskal–Wallis
                non_empty = [g.dropna().astype(float) for g in groups if g.dropna().shape[0] > 0]
                if len(non_empty) >= 2:
                    stat, p = stats.kruskal(*non_empty)
                else:
                    stat, p = np.nan, np.nan
                stat_s, p_s = fmt_stat_p(stat, p)
                row["Statistic"] = stat_s
                row["P-value"] = p_s
                
                # store raw p for correction
                raw_pvals.append(p)
                row_indices.append(len(rows))   # store row index
                rows.append(row)

        else:
            # categorical (multi- or binary)
            # For each group, show n (%) where var is not NA (for denom) and for *all* levels combined.
            # If binary -> display count of the "positive" (second) level; if multi -> display total non-missing n (% of table N).
            gcat = d[[group_col, var]].copy()
            # Determine binary vs multi
            levels = gcat[var].dropna().unique()
            if gcat[var].dtype == "bool" or (len(levels) <= 2):
                # choose the "positive" level: True if boolean, else the 2nd sorted level
                if gcat[var].dtype == "bool":
                    pos_mask = gcat[var] == True
                    pos_label = "Yes"
                else:
                    lvls_sorted = pd.Series(levels).sort_values().tolist()
                    pos = lvls_sorted[-1] if len(lvls_sorted) == 2 else lvls_sorted[0]
                    pos_mask = gcat[var] == pos
                    pos_label = str(pos)

                # per-group: n_pos (% of non-missing in that group)
                for c in group_order:
                    sub = gcat.loc[gcat[group_col] == c, var]
                    denom = sub.notna().sum()
                    n_pos = (sub == (True if gcat[var].dtype == "bool" else pos)).sum()
                    row[c] = fmt_n_pct(n_pos, denom if denom > 0 else np.nan)

                # χ² test on contingency (countries x pos/neg)
                ct = pd.crosstab(gcat[group_col], pos_mask)
                if ct.shape[0] > 1 and ct.shape[1] > 1:
                    stat, p, _, _ = stats.chi2_contingency(ct)
                else:
                    stat, p = np.nan, np.nan
                stat_s, p_s = fmt_stat_p(stat, p)
                row["Statistic"] = stat_s
                row["P-value"] = p_s
                
                # store raw p for correction
                raw_pvals.append(p)
                row_indices.append(len(rows))   # store row index
                rows.append(row)

                # rename parameter to include the positive level 
                row["Parameter"] = f"{var}"  

            else:
                # Multi-category: show total non-missing n (%) per group relative to table N
                total_n = len(d)
                for c in group_order:
                    sub = gcat.loc[gcat[group_col] == c, var]
                    denom = sub.notna().sum()
                    row[c] = fmt_n_pct(denom, total_n)

                # χ² test across all levels
                ct = pd.crosstab(gcat[group_col], gcat[var])
                if ct.shape[0] > 1 and ct.shape[1] > 1:
                    stat, p, _, _ = stats.chi2_contingency(ct)
                else:
                    stat, p = np.nan, np.nan
                stat_s, p_s = fmt_stat_p(stat, p)
                row["Statistic"] = stat_s
                row["P-value"] = p_s
                
                # store raw p for correction
                raw_pvals.append(p)
                row_indices.append(len(rows))   # store row index
                rows.append(row)

    # ------------------------------
    #   Apply Benjamini–Hochberg FDR
    # ------------------------------
    raw_pvals_clean = [p if p is not None and not np.isnan(p) else 1.0 
                       for p in raw_pvals]

    reject, qvals, _, _ = multipletests(raw_pvals_clean, method="fdr_bh")

    # insert q-values back into rows
    for idx, q in zip(row_indices, qvals):
        rows[idx]["Q-value"] = f"{q:.{digits_p}g}"

    # Build final DataFrame
    cols = ["Parameter"] + group_order + ["Statistic", "Q-value"]
    out = pd.DataFrame(rows)[cols]

    return out


In [ ]:
md['Diagnosis'].value_counts()


In [ ]:
variables = ["#N", 'Age', 'BMI', 'Sex: Female', 'Race: White', 'Race: Black or African American', 'Race: Other', 'Ethnicity: Hispanic', 'MOCA', 'Depression', 'Anxiety', 'Antidepressants',
'pTau181', 'pTau217', 'Abeta40', 'Abeta42', 'HEI2015Score']

var_types = {
    "Age": "continuous",
    "BMI": "continuous",
    "Sex: Female": "categorical",
    'Race: White': "categorical", 
    'Race: Black or African American': "categorical", 
    'Race: Other': "categorical", 
    'Ethnicity: Hispanic': "categorical",
    'MOCA': "continuous", 
    'Depression':"categorical", 
    'Antibiotic History: Within the Last Month':"categorical", 
    'Anxiety': "categorical",
    'Antidepressants': "categorical", 
    'pTau181': "continuous", 
    'pTau217': "continuous",
    'Abeta40': "continuous",
    'Abeta42': "continuous",
    'HEI2015Score': "continuous"
}

table1 = make_group_summary_table(
    md,
    group_col="Diagnosis",
    variables=variables,
    var_types=var_types,
    group_order=["Cognitively Unimpaired", "Cognitively Impaired"],
    continuous_test="kruskal",  
    digits_mean=1, digits_sd=1, digits_pct=1, digits_stat=2, digits_p=2
)


In [ ]:
# Save or display
table1.to_csv("demographics_table.csv", index=False)
